# NYC Taxi Trips - Basics

**Dataset:** `samples.nyctaxi.trips`

**Difficulty:** Easy

**Topics:** filter, count, aggregation, sorting

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Loading, Filtering, and Aggregating

| Function | What it does |
|----------|-------------|
| `spark.table("catalog.schema.table")` | Loads a Delta/Unity Catalog table into a DataFrame |
| `df.count()` | Triggers compute and returns the total number of rows (action) |
| `df.filter(condition)` | Returns rows matching the condition (transformation — lazy) |
| `df.groupBy(col).agg(...)` | Groups rows by a column and applies aggregate functions |
| `df.orderBy(col.desc())` | Sorts rows in descending order |
| `F.col("name")` | References a column by name |
| `F.avg()`, `F.min()`, `F.max()`, `F.count()` | Aggregation functions |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html) · [Databricks PySpark](https://docs.databricks.com/en/pyspark/index.html)

> **Transformations** (like `filter`, `groupBy`) are lazy — they build a plan but do no work.
> **Actions** (like `count`, `show`, `collect`) trigger compute and return results.

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.nyctaxi.trips")

# Action: how many rows exist?
print("Total rows:", df.count())

# Transformation (lazy): filter to vendor 1
vendor1 = df.filter(F.col("vendor_id") == "1")

# Action: compute stats by vendor
df.groupBy("vendor_id").agg(
    F.count("*").alias("num_trips"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare")
).orderBy(F.col("num_trips").desc()).show()

In [0]:
display(df)

## Problem 1

Count the **total number of trips** in the NYC taxi dataset.
Load the table `samples.nyctaxi.trips` and return a single-row DataFrame
that tells us how many records exist.

**Expected output columns:**
- `total_trips` (bigint) - total number of trip records

In [0]:
# Problem 1 - write your solution here 
result_1 = spark.table("samples.nyctaxi.trips")
# Assign your result to: result_1 
result_1 = result_1.agg(F.count("*").alias("total_trips"))

display(result_1)

In [0]:
display(result_1)

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'total_trips' in cols, "Missing column: total_trips"
cnt = result_1.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
total = result_1.collect()[0]['total_trips']
assert total > 0, f"Expected total_trips > 0, got {total}"
assert total < 10_000_000, f"total_trips seems unreasonably large: {total}"
print(f"Problem 1 passed ✓  ({cnt} rows, total_trips={total})")

## Problem 2

Find all trips where the **trip distance is greater than 10 miles**.
Return the full original row for each qualifying trip.

**Expected output columns:**
- `tpep_pickup_datetime` - pickup timestamp
- `tpep_dropoff_datetime` - dropoff timestamp
- `trip_distance` - distance in miles (must be > 10)
- `fare_amount` - fare charged
- `pickup_zip` - pickup ZIP code
- `dropoff_zip` - dropoff ZIP code

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2 
df = spark.table("samples.nyctaxi.trips") 
result_2 = df.filter(F.col("trip_distance")>10)

display(result_2)  # replace this

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'trip_distance' in cols, "Missing column: trip_distance"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_dist = result_2.agg(F.min('trip_distance')).collect()[0][0]
assert min_dist > 10, f"All trip_distance values must be > 10, found min={min_dist}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Calculate summary statistics for the **fare amount** across all trips.
Return a single-row DataFrame with the average, minimum, and maximum fare.

**Expected output columns:**
- `avg_fare` - average fare amount across all trips
- `min_fare` - minimum fare amount
- `max_fare` - maximum fare amount

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3 
result_3 = df.agg(F.avg('fare_amount').alias('avg_fare'),
                  F.min('fare_amount').alias('min_fare'),
                  F.max('fare_amount').alias('max_fare')
)


display(result_3)  # replace this

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'avg_fare' in cols, "Missing column: avg_fare"
assert 'min_fare' in cols, "Missing column: min_fare"
assert 'max_fare' in cols, "Missing column: max_fare"
cnt = result_3.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_3.collect()[0]
assert row['max_fare'] >= row['avg_fare'] >= row['min_fare'], "Expected min_fare <= avg_fare <= max_fare"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find the **top 5 pickup ZIP codes** by the number of trips originating there.
Sort the results in descending order of trip count.

**Expected output columns:**
- `pickup_zip` - the ZIP code where trips started
- `trip_count` - number of trips from that ZIP code (sorted descending)

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4 
result_4 = df.groupBy('pickup_zip').agg(F.count('*').alias('trip_count')).orderBy(F.desc('trip_count')).limit(5)
display(result_4)  # replace this

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
cnt = result_4.count()
assert cnt == 5, f"Expected exactly 5 rows (top 5), got {cnt}"
counts = [r['trip_count'] for r in result_4.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by trip_count descending"
assert all(c > 0 for c in counts), "All trip counts must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Count the number of trips per **dropoff ZIP code**, but only include trips
where the fare amount is **greater than $20**.
This helps identify which areas attract higher-value trips.

**Expected output columns:**
- `dropoff_zip` - the ZIP code where trips ended
- `trip_count` - number of qualifying trips (fare > $20) ending at that ZIP

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
result_5 = df.filter(F.col('fare_amount') > 20).groupBy('dropoff_zip').agg(F.count('*').alias('trip_count'))
display(result_5)  # replace this

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'dropoff_zip' in cols, "Missing column: dropoff_zip"
assert 'trip_count' in cols, "Missing column: trip_count"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['trip_count'] for r in result_5.collect()]
assert all(c > 0 for c in counts), "All trip_count values must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")